### Counting annotated abfs and total abfs seeing which files my App Script failed to download

- 158 abfs are annotated in `4D...` excel sheet out of 794 in the shared Google Folder
- 29 out of 39 cells are labeled in the `List of Cells` excel sheet 

In [2]:
# parent_folder_path = "/Users/haleyoro/Desktop/" # work on library computer
parent_folder_path = "/Users/Haley/Desktop/" # work on local computer


In [3]:
# Imports
import pandas as pd
import numpy as np
import re
import pyabf

### Names of each annotation sheet

In [4]:
sheet_names_df = pd.read_csv(parent_folder_path+'/murray-neuroscience-lab/New processed excels/sheet_names.csv', header=None)
sheet_names = sheet_names_df.iloc[:,0].values
print(len(sheet_names)==len(set(sheet_names)))
print(len(sheet_names))

sheet_names # names of each annotation sheet

True
39


array(['2012_04_25_cell1', '2012_04_25_cell3', '2012_04_27_cell1', ...,
       '2013_03_22_cell5', '2013_03_22_cell6', '2012_10_04_cell1'],
      shape=(39,), dtype=object)

### Each sheet is a different cell , so all traces in one sheet are the same cell type and fast/slow preferrring population

In [4]:
cell_types_df = pd.read_csv(parent_folder_path+"/murray-neuroscience-lab/Excel processor/List of cells.csv")
cell_types_df = cell_types_df.dropna(how='all')
cell_types_df = cell_types_df.iloc[:,:4]
cell_types_df.reset_index(drop=True,inplace=True)
cell_types_df

,Cell,Cell Type,Input Resistance,Motoneuron
0,2013_03_21_cell5,CaP,33.00,primary
1,2013_03_22_cell6,vRoP,42.00,primary
2,2013_03_22_cell2,CaP,43.64,primary
3,2012_08_31_cell2,vRoP,44.77,primary
4,2012_08_31_cell1,MiP,45.33,primary
5,2012_08_31_cell3,MiP,46.67,primary
6,2013_03_21_cell4,MiP,47.15,primary
7,2012_08_01_cell3,MiP,49.99,primary
8,2013_03_22_cell1,CaP,51.00,primary
9,2013_03_20_cell1,MiP,51.10,primary


In [5]:
labled_cells = cell_types_df["Cell"].to_numpy()

# There are 10 sheets that arent labled

In [6]:
# Define two lists
list1 = labled_cells
list2 = sheet_names

# Find extra elements in list1
extra_in_list1 = list(set(list1) - set(list2))

# Find extra elements in list2
extra_in_list2 = list(set(list2) - set(list1))

# Print the results
print("Extra elements in list1:", extra_in_list1)
print("Extra elements in list2:", extra_in_list2)
len(extra_in_list2)

Extra elements in list1: []
Extra elements in list2: ['2013_03_21_cell2', '2012_12_04_cell2', '2012_12_06_cell2', '2012_04_27_cell1', '2012_06_25_cell3', '2012_12_06_cell6', '2012_06_22_cell3', '2012_10_04_cell1', '2012_12_06_cell5', '2012_12_06_cell3']


10

### All abf files available in shared folder

In [14]:
allabfs = pd.read_csv(parent_folder_path+'murray-neuroscience-lab/Excel processor/all_abf_files.csv',header=None)
allabfs = allabfs[0].to_numpy()
allabfs 

array(['2013_03_22_0045.abf', '2013_03_22_0042.abf',
       '2013_03_22_0039.abf', ..., '2012_08_29_0009.abf',
       '2012_08_29_0016.abf', '2012_08_29_0018.abf'],
      shape=(1050,), dtype=object)

Check to see duplicates

In [18]:
pattern = r"\d{4}_\d{2}_\d{2}_\d{4}\.abf"

allabfs = [f for f in allabfs if re.fullmatch(pattern, f)]

print(len(allabfs)) 
print(len(set(allabfs)))

1047
794


In [13]:
moreabfs = pd.read_csv(parent_folder_path+'murray-neuroscience-lab/Excel processor/more_abfs.csv')
moreabfs = moreabfs["Missing files"].to_numpy()
moreabfs 

array(['2013_03_22_0054.abf', '2012_12_04_0016.abf',
       '2012_12_06_0052.abf', ..., '2013_03_22_0041.abf',
       '2013_03_22_0056.abf', '2012_12_06_0059.abf'],
      shape=(84,), dtype=object)

In [5]:
abfs = pd.read_csv(parent_folder_path+'murray-neuroscience-lab/Excel processor/abf_trace_names.csv')
abfs = abfs["filename"].to_numpy()
abfs 

array(['2012_04_25_0004.abf', '2012_04_25_0005.abf',
       '2012_04_25_0006.abf', ..., '2013_03_22_0060.abf',
       '2013_03_22_0061.abf', '2012_10_04_0014.abf'],
      shape=(158,), dtype=object)

In [22]:
abfstraces = np.concatenate((moreabfs,abfs))
len(abfstraces)

242

In [6]:
all_trace_names = []
for sheet in sheet_names:
    file_path = parent_folder_path + "murray-neuroscience-lab/New processed excels/"+ sheet + '.csv'
    df_filtered = pd.read_csv(file_path)
    trace_names = df_filtered["Trace name"].unique().tolist()
    all_trace_names.extend(trace_names)

print(len(all_trace_names)==len(set(all_trace_names)))
print(len(all_trace_names))
print(type(all_trace_names[0]))


True
242
<class 'str'>


In [36]:
abf_names = [name + ".abf" for name in all_trace_names]
len(abf_names)

242

Yay items match!

In [26]:
# Define two lists
list1 = abf_names
list2 = abfstraces

# Find extra elements in list1
extra_in_list1 = list(set(list1) - set(list2))

# Find extra elements in list2
extra_in_list2 = list(set(list2) - set(list1))

# Print the results
print("Extra elements in list1:", extra_in_list1)
print("Extra elements in list2:", extra_in_list2)

Extra elements in list1: []
Extra elements in list2: []


Note: only 242 out of 794 abf files have annotations 

In [7]:
df = pd.DataFrame(all_trace_names)
df.to_csv("all_trace_names.csv", index=False, header=False)

Check to see that all the sheet traces have abf files

In [ ]:
file_path = ""
df = pd.read_csv(file_path)
df[["Trace name","Tags","Type"]] = df[["Trace name","Tags","Type"]].astype("string")
traces = df["Trace name"].unique().tolist()
abf = {}
for trace in trace_names:
    abfs[trace] = []
    file_path2 = "/Users/Haley/Downloads/.abf files annotated/" + trace + ".abf"
    abf = pyabf.ABF(file_path2)
    abfs[trace].append(abf)
    